In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout, Add, BatchNormalization, Activation
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt


# 加载数据，这里假设你有包含'text'（新闻文本）和'label'（新闻真假标签）列的数据
train_df = pd.read_excel('E:/Fakenews Detection/Dataest_2.0/Fakenews_train.xlsx', engine='openpyxl')
test_df = pd.read_excel('E:/Fakenews Detection/Dataest_2.0/Fakenews_test.xlsx', engine='openpyxl')

# 提取文本和标签
X_train_texts = train_df['text'].values
y_train = train_df['label'].values
X_test_texts = test_df['text'].values
y_test = test_df['label'].values

# 预处理文本数据
max_words = 10000
max_len = 150

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(X_train_texts)
X_train_sequences = tokenizer.texts_to_sequences(X_train_texts)
X_test_sequences = tokenizer.texts_to_sequences(X_test_texts)

X_train = pad_sequences(X_train_sequences, maxlen=max_len)
X_test = pad_sequences(X_test_sequences, maxlen=max_len)

# 将标签转换为浮点数
y_train = y_train.astype(float)
y_test = y_test.astype(float)


def resnet_block(x, filters, kernel_size=3, stride=1):
    y = Conv1D(filters, kernel_size, strides=stride, padding='same')(x)
    y = BatchNormalization()(y)
    y = Activation('relu')(y)

    y = Conv1D(filters, kernel_size, strides=1, padding='same')(y)
    y = BatchNormalization()(y)

    if stride!= 1 or x.shape[-1]!= filters:
        x = Conv1D(filters, kernel_size=1, strides=stride, padding='same')(x)
    y = Add()([x, y])
    y = Activation('relu')(y)
    return y


def build_resnet_model(max_words, max_len, embedding_dim=100, num_blocks=5, filters=128):
    """
    构建ResNet模型结构
    """
    inputs = Input(shape=(max_len,))
    # 随机初始化词嵌入矩阵
    embedding_layer = Embedding(max_words, embedding_dim)(inputs)

    x = Conv1D(filters, 7, strides=2, padding='same')(embedding_layer)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    for _ in range(num_blocks):
        x = resnet_block(x, filters)

    x = GlobalMaxPooling1D()(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.5)(x)
    outputs = Dense(1, activation='sigmoid')(x)

    model = Model(inputs, outputs)
    return model


def generate_adversarial_samples(model, X_sample, y_sample, epsilon=0.1):
    X_sample_tensor = tf.convert_to_tensor(X_sample, dtype=tf.int32)
    y_sample_tensor = tf.convert_to_tensor(y_sample, dtype=tf.float32)
    # 用于记录模型前向传播时的操作，以便之后进行反向传播计算梯度。
    with tf.GradientTape() as tape:
        embed_output = model.layers[1](X_sample_tensor)
        tape.watch(embed_output)
        batch_size = tf.shape(embed_output)[0]
        sequence_length = tf.shape(embed_output)[1]
        embedding_dim = tf.shape(embed_output)[2]
        # 先将维度展平为 (batch_size, sequence_length * embedding_dim)
        flatten_output = tf.reshape(embed_output, [batch_size, -1])

        # 确保传递给全连接层的维度符合要求，这里根据模型结构调整了处理方式
        features_dim = sequence_length * embedding_dim
        if features_dim!= 256:
            # 如果维度不一致，添加一个适配层（可根据实际情况调整）
            flatten_output = Dense(256, activation='relu')(flatten_output)

        predictions = model.layers[-1](model.layers[-2](flatten_output))
        loss = tf.keras.losses.BinaryCrossentropy()(y_sample_tensor, predictions)

    gradients = tape.gradient(loss, embed_output)  # 计算嵌入层输出相对于损失的梯度。
    perturbations = epsilon * tf.sign(gradients)  # 根据梯度生成扰动。epsilon 控制扰动强度，sign 函数获取梯度的符号

    embed_output_adv = embed_output + perturbations

    # 将对抗扰动后的结果再转换回合适的维度，这里假设和原始 embed_output 维度一致
    embed_output_adv = tf.reshape(embed_output_adv, [batch_size, sequence_length, embedding_dim])
    adversarial_samples = tf.argmax(embed_output_adv, axis=-1).numpy()
    return adversarial_samples


def train_and_evaluate_models(X_train, y_train, X_test, y_test, max_words, max_len):
    """
    训练并评估常规模型和对抗训练模型
    """
    # 常规模型
    model_regular = build_resnet_model(max_words, max_len)
    model_regular.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
                          loss=tf.keras.losses.BinaryCrossentropy(), metrics=['accuracy'])

    # 模型训练
    history_regular = model_regular.fit(X_train, y_train, epochs=10, batch_size=8, validation_split=0.2, verbose=1)

    # 评估常规模型在原始测试集上的性能
    y_pred_regular = model_regular.predict(X_test)
    y_pred_regular = (y_pred_regular > 0.5).astype(int).ravel()
    regular_acc = accuracy_score(y_test, y_pred_regular)
    regular_precision = precision_score(y_test, y_pred_regular)
    regular_recall = recall_score(y_test, y_pred_regular)
    regular_f1 = f1_score(y_test, y_pred_regular)
    print(f'常规模型在原始测试集上的准确率: {regular_acc:.4f}，精确率: {regular_precision:.4f}，召回率: {regular_recall:.4f}，F1分数: {regular_f1:.4f}')

    # 生成对抗样本并评估常规模型在对抗样本上的表现
    X_test_adv = generate_adversarial_samples(model_regular, X_test, y_test)
    y_pred_regular_adv = model_regular.predict(X_test_adv)
    y_pred_regular_adv = (y_pred_regular_adv > 0.5).astype(int).ravel()
    regular_adv_acc = accuracy_score(y_test, y_pred_regular_adv)
    regular_adv_precision = precision_score(y_test, y_pred_regular_adv)
    regular_adv_recall = recall_score(y_test, y_pred_regular_adv)
    regular_adv_f1 = f1_score(y_test, y_pred_regular_adv)
    print(f'常规模型在对抗样本上的准确率: {regular_adv_acc:.4f}，精确率: {regular_adv_precision:.4f}，召回率: {regular_adv_recall:.4f}，F1分数: {regular_adv_f1:.4f}')

    # 对抗性训练模型
    model_adv = build_resnet_model(max_words, max_len)
    model_adv.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
                      loss=tf.keras.losses.BinaryCrossentropy(), metrics=['accuracy'])

    # 生成训练集的对抗样本
    X_train_adv = generate_adversarial_samples(model_adv, X_train, y_train)

    # 合并原始训练样本和对抗样本进行对抗训练
    X_combined_train = np.concatenate((X_train, X_train_adv), axis=0)
    y_combined_train = np.concatenate((y_train, y_train), axis=0)

    history_adv = model_adv.fit(X_combined_train, y_combined_train, epochs=10, batch_size=8, validation_split=0.2, verbose=1)

    # 评估对抗性训练模型在原始测试集上的表现
    y_pred_adv = model_adv.predict(X_test)
    y_pred_adv = (y_pred_adv > 0.5).astype(int).ravel()
    adv_acc = accuracy_score(y_test, y_pred_adv)
    adv_precision = precision_score(y_test, y_pred_adv)
    adv_recall = recall_score(y_test, y_pred_adv)
    adv_f1 = f1_score(y_test, y_pred_adv)
    print(f'对抗性训练模型在原始测试集上的准确率: {adv_acc:.4f}，精确率: {adv_precision:.4f}，召回率: {adv_recall:.4f}，F1分数: {adv_f1:.4f}')

    # 评估对抗性训练模型在对抗样本上的表现
    y_pred_adv_adv = model_adv.predict(X_test_adv)
    y_pred_adv_adv = (y_pred_adv_adv > 0.5).astype(int).ravel()
    adv_adv_acc = accuracy_score(y_test, y_pred_adv_adv)
    adv_adv_precision = precision_score(y_test, y_pred_adv_adv)
    adv_adv_recall = recall_score(y_test, y_pred_adv_adv)
    adv_adv_f1 = f1_score(y_test, y_pred_adv_adv)
    print(f'对抗性训练模型在对抗样本上的准确率: {adv_adv_acc:.4f}，精确率: {adv_adv_precision:.4f}，召回率: {adv_adv_recall:.4f}，F1分数: {adv_adv_f1:.4f}')

    return history_regular, history_adv

train_and_evaluate_models(X_train, y_train, X_test, y_test, max_words, max_len)


d:\conda\tensorflow-gpu2.2.0\lib\site-packages\requests\__init__.py:104: RequestsDependencyWarning: urllib3 (1.26.20) or chardet (5.0.0)/charset_normalizer (2.0.12) doesn't match a supported version!
  RequestsDependencyWarning)


Epoch 1/10
994/994 [==============================] - 64s 65ms/step - loss: 0.6969 - accuracy: 0.7479 - val_loss: 0.4861 - val_accuracy: 0.8068
Epoch 2/10
994/994 [==============================] - 64s 64ms/step - loss: 0.4011 - accuracy: 0.8283 - val_loss: 0.4982 - val_accuracy: 0.8084
Epoch 3/10
994/994 [==============================] - 64s 64ms/step - loss: 0.2343 - accuracy: 0.9129 - val_loss: 0.5347 - val_accuracy: 0.8038
Epoch 4/10
994/994 [==============================] - 64s 64ms/step - loss: 0.1032 - accuracy: 0.9664 - val_loss: 0.5848 - val_accuracy: 0.7731
Epoch 5/10
994/994 [==============================] - 65s 65ms/step - loss: 0.0508 - accuracy: 0.9853 - val_loss: 0.6200 - val_accuracy: 0.7706
Epoch 6/10
994/994 [==============================] - 65s 65ms/step - loss: 0.0391 - accuracy: 0.9897 - val_loss: 0.6625 - val_accuracy: 0.7736
Epoch 7/10
994/994 [==============================] - 63s 63ms/step - loss: 0.0335 - accuracy: 0.9917 - val_loss: 0.8274 - val_accuracy:

(<tensorflow.python.keras.callbacks.History at 0x217242359e8>,
 <tensorflow.python.keras.callbacks.History at 0x2172f8179b0>)